In [1]:
import os
import warnings

from datetime import datetime, timezone

import hopsworks
import numpy as np
import pandas as pd
import polars as pl

from dotenv import load_dotenv
from hopsworks.project import Project
from hsfs.feature_group import FeatureGroup
from hsfs.feature_store import FeatureStore
from omegaconf import OmegaConf

from src.config import Config, load_config
from src.ingest import download_file

warnings.filterwarnings("ignore")
load_dotenv(Config.HOME_DIR / ".env")

True

In [2]:
# specify the starting and ending timestamps for which to ingest data
current_ts: pd.Timestamp = pd.Timestamp(datetime.now(timezone.utc)).floor("H")
end: pd.Timestamp = current_ts - pd.Timedelta(days=366)
start: pd.Timestamp = end - pd.Timedelta(days=28)

# a list of pre-processed pd.DataFrames, one per (year, month) pair
dfs: list[pd.DataFrame] = [
    download_file(year, month) 
    for year, month in zip([start.year, end.year], [start.month, end.month])
]

# (1) convert each pd.DataFrame in the 'dfs' list to a pl.DataFrame, and ...
# vertically concatenate them into a single pl.DataFrame
# (2) modify the 'pickup_datetime' column's timestamps by setting their timezone to UTC and ...
# offsetting them forward in time by one year.  Then, filter the pl.DataFrame by only keeping ...
# records (rows) whose modified timestamp is greater than or equal to the 'start' timestamp and ...
# less than or equal to the 'end' timestamp
# (3) modify the 'pickup_datetime' column's timestamps by ...
# (a) setting their timezone to UTC, ...
# (b) offsetting them forward in time by one year, and ...
# (c) converting them to integers that represent the time passed, in ms, since the Unix EPOCH
# (4) rename the 'pickup_datetime' column to 'unix_epoch_ms'
# (5) convert the pl.DataFrame to a pd.DataFrame
# NOTE: the NYC taxi data lags three months behind the current date, so the timestamps are ...
# offset forward in time to simulate a data ingestion scenario that occurs in real time
# NOTE: the 'df' pd.DataFrame will be written to Hopsworks
df: pd.DataFrame = (
    pl.concat([pl.from_pandas(df) for df in dfs], how="vertical") # (1)
    .filter( # (2)
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").ge(start),
        pl.col("pickup_datetime").dt.replace_time_zone("UTC").le(end)
    )
    .with_columns( # (3)
        pl.col("pickup_datetime")
        .dt.replace_time_zone("UTC")
        .dt.offset_by("1y")
        .dt.epoch(time_unit="ms")
    )
    .rename({"pickup_datetime": "unix_epoch_ms"}) # (4)
    .to_pandas() # (5)
)
df

2024-07-16 10:25:40,647 INFO: Downloading, validating, and pre-processing https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-06.parquet


100%|██████████| 258/258 [00:04<00:00, 58.62it/s]


2024-07-16 10:25:53,953 INFO: Downloading, validating, and pre-processing https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet


100%|██████████| 255/255 [00:03<00:00, 65.83it/s]


,location_id,unix_epoch_ms,rides
0,1,1718730000000,0.0
1,1,1718733600000,2.0
2,1,1718737200000,0.0
3,1,1718740800000,0.0
4,1,1718744400000,1.0
...,...,...,...
161488,265,1721134800000,3.0
161489,265,1721138400000,4.0
161490,265,1721142000000,1.0
161491,265,1721145600000,11.0


In [3]:
HOPSWORKS_CONFIG: dict[str, list[str] | str] = OmegaConf.to_container(load_config().hopsworks)
HOPSWORKS_CONFIG

{'project_name': 'taxi_demand_forecasting',
 'feature_group_name': 'univariate_time_series',
 'primary_key': ['location_id', 'unix_epoch_ms'],
 'event_time': 'unix_epoch_ms'}

In [4]:
# connect to the Hopsworks 'taxi_demand_forecasting' project
project: Project = hopsworks.login(
    project=HOPSWORKS_CONFIG.get("project_name"),
    api_key_value=os.getenv("HOPSWORKS_API_KEY")
)

# connect to the project's feature store
feature_store: FeatureStore = project.get_feature_store()

# create the feature store's feature group
# NOTE: the feature group allows the 'df' pd.DataFrame's features to be saved to the feature store
# NOTE: the 'primary_key' parameter below, which can be set equal to a single column or ...
# a list of columns from the 'df' pd.DataFrame, is used as a unique identifier for each record (row)
feature_group: FeatureGroup = feature_store.get_or_create_feature_group(
    name=HOPSWORKS_CONFIG.get("feature_group_name"),
    version=1,
    description="NYC taxi rides recorded at an hourly frequency",
    primary_key=HOPSWORKS_CONFIG.get("primary_key"),
    event_time=HOPSWORKS_CONFIG.get("event_time")
)

# save the 'df' pd.DataFrame to the 'taxi_demand_forecasting' project as a new feature group
# NOTE: the feature group's name is, 'univariate_time_series', and it can be found on the ...
# Hopsworks UI, under the 'taxi_demand_forecasting' project's 'Feature Group' tab 
feature_group.insert(df, write_options={"wait_for_job": False})

Connected. Call `.close()` to terminate connection gracefully.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/708756
Connected. Call `.close()` to terminate connection gracefully.
Feature Group created successfully, explore it at 
https://c.app.hopsworks.ai:443/p/708756/fs/704579/fg/1003873


Uploading Dataframe: 0.00% |          | Rows 0/161493 | Elapsed Time: 00:00 | Remaining Time: ?

Launching job: univariate_time_series_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai/p/708756/jobs/named/univariate_time_series_1_offline_fg_materialization/executions


(<hsfs.core.job.Job at 0x307d9e7a0>, None)